In [2]:
# -*- coding: utf-8 -*-
"""
Created on Fri 26/09/2024

@author: Adapted by Carlos Javier Navarro
"""

# Sources:
    # https://platform.openai.com/docs/guides/batch/rate-limits
    # https://cookbook.openai.com/examples/batch_processing
    # https://cookbook.openai.com/examples/tag_caption_images_with_gpt4v (to check)

# Make sure you have the latest version of the SDK available to use the Batch API
# pip3 install openai --upgrade

import json
from openai import OpenAI
import pandas as pd
from IPython.display import Image, display

OPENAI_API_KEY='sk-svcacct-jLSnF1FIehnmGBvWCayuT3BlbkFJ1ARy381duuMQxcc5ejUH'

# Initializing OpenAI client - see https://platform.openai.com/docs/quickstart?context=python
client = OpenAI(api_key=OPENAI_API_KEY)

dataset_path = "listado_imagenes2.csv" # furniture_amazon_dataset.csv
df = pd.read_csv(dataset_path)

df['id2'] = df['id2'].astype(str)

df.head()

# Rhoaifa_new prompt 1
caption_system_prompt = '''
Classify the image into one of these categories: Cultural_Religious, Fauna_Flora, Gastronomy, Nature, Sports, or Urban_Rural, I just need you to give me the name of the class as a result without any extra explanation. 
'''


def get_caption(img_url, title):
    response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.2,
    max_tokens=300,
    messages=[
        {
            "role": "system",
            "content": caption_system_prompt
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": title
                },
                # The content type should be "image_url" to use gpt-4-turbo's vision capabilities
                {
                    "type": "image_url",
                    "image_url": {
                        "url": img_url
                    }
                },
            ],
        }
    ]
    )

    return response.choices[0].message.content


# Testing the prompt on a few images
for _, row in df[:5].iterrows():
    img_url = row['url'] # field with url of the image (primary_image)
    caption = get_caption(img_url, row['id2']) # title
    img = Image(url=img_url)
    display(img)
    print(row['id2'])
    print(f"CAPTION: {caption}\n\n")


# Creating an array of json tasks

tasks = []

for index, row in df.iterrows():
    
    title = row['id2']
    img_url = row['url']
    
    task = {
        "custom_id": f"task-{index}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # This is what you would have in your Chat Completions API call
            "model": "gpt-4o-mini",
            "temperature": 0.2,
            "max_tokens": 300,
            "messages": [
                {
                    "role": "system",
                    "content": caption_system_prompt
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": title
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": img_url
                            }
                        },
                    ],
                }
            ]            
        }
    }
    
    tasks.append(task)    

# Creating the file

file_name = "level3_earthcul_CN_adj.jsonl"

with open(file_name, 'w') as file:
    for obj in tasks:
        file.write(json.dumps(obj) + '\n')

# Uploading the file 

batch_file = client.files.create(
  file=open(file_name, "rb"),
  purpose="batch"
)

10060247786
CAPTION: Fauna_Flora




10175081236
CAPTION: Fauna_Flora




11162720455
CAPTION: Fauna_Flora




13807137553
CAPTION: Fauna_Flora




In [3]:
# Creating the batch job
batch_job = client.batches.create(
  input_file_id=batch_file.id,
  endpoint="/v1/chat/completions",
  completion_window="24h"
)